# 02 · Flujo de compra end-to-end · saucedemo (PROD)

Login → catálogo → carrito → checkout → confirmación, validando los montos.

Ejecuta la celda de código. Debajo aparece el log con fecha (dd-mm-aaaa), hora y ambiente. El mismo log queda en `reports/logs/` y las capturas en `reports/evidencias/`. Chrome se abre en modo visible (para ocultarlo: variable de entorno `HEADLESS=1`).

In [ ]:
import contextlib
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # permite importar pages/ y utils/

from pages import InventoryPage, LoginPage
from utils.config import AMBIENTE, BASE_URL, PASSWORD, USERS
from utils.driver_factory import create_driver
from utils.evidence import save_screenshot
from utils.logger import get_logger, setup_execution_log

log_file = setup_execution_log("02_flujo_compra_e2e")
log = get_logger()
log.info("===== INICIO | Flujo de compra end-to-end =====")
log.info("Ambiente: %s | Sitio: %s", AMBIENTE, BASE_URL)

PRODUCTOS = ["Sauce Labs Backpack", "Sauce Labs Fleece Jacket"]
COMPRADOR = ("Ricardo", "Pavez", "8320000")

driver = create_driver()
try:
    # 1. Login
    inventory = LoginPage(driver).load().login_as(USERS["standard"])
    log.info("[OK] Login exitoso como %s", USERS["standard"])

    # 2. Catálogo ordenado por precio
    inventory.sort_by(InventoryPage.SORT_PRICE_ASC)
    for nombre, precio in zip(inventory.product_names(), inventory.product_prices()):
        log.info("Producto: $%6.2f  %s", precio, nombre)

    # 3. Carrito
    inventory.add_to_cart(*PRODUCTOS)
    assert inventory.cart_count() == len(PRODUCTOS), f"Badge del carrito: {inventory.cart_count()}"
    log.info("[OK] Badge del carrito = %d", inventory.cart_count())
    cart = inventory.go_to_cart()
    assert sorted(cart.item_names()) == sorted(PRODUCTOS), f"Carrito: {cart.item_names()}"
    log.info("[OK] Productos en el carrito: %s", ", ".join(cart.item_names()))
    save_screenshot(driver, "02_carrito")

    # 4. Checkout y validación de montos
    overview = cart.checkout().continue_to_overview(*COMPRADOR)
    suma_precios = round(sum(overview.item_prices()), 2)
    subtotal, impuesto, total = overview.subtotal(), overview.tax(), overview.total()
    log.info("Suma de precios $%.2f | Subtotal $%.2f | Impuesto $%.2f | Total $%.2f",
             suma_precios, subtotal, impuesto, total)
    assert suma_precios == subtotal, "El subtotal no coincide con la suma de precios"
    assert round(subtotal + impuesto, 2) == total, "El total no es subtotal + impuesto"
    log.info("[OK] Montos del resumen correctos")
    save_screenshot(driver, "02_resumen_compra")

    # 5. Confirmación
    complete = overview.finish()
    assert complete.header() == "Thank you for your order!", f"Mensaje final: {complete.header()}"
    log.info("[OK] Compra confirmada -> '%s'", complete.header())
    save_screenshot(driver, "02_compra_confirmada")

    log.info("RESULTADO: todas las validaciones OK")
except Exception:
    log.exception("RESULTADO: la ejecución falló")
    with contextlib.suppress(Exception):
        save_screenshot(driver, "02_error")
    raise
finally:
    driver.quit()
    log.info("===== FIN | Log guardado en %s =====", log_file.name)